# 9. Describing a dataset: summaries and Claude: exercises

Exercises of [chapter 9](../09_describing_a_dataset.md). Run the setup cell first.

Work in a copy of this notebook: it is regenerated by `docs/manual/tools/build_notebooks.py`.

In [ ]:
# Setup: find the repository root, make the sample data root if needed, use the sample's site registry
import os, subprocess, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))
SAMPLE = ROOT / "output/manual/sample_root"
if not SAMPLE.exists():
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"
EXTRACT = ROOT / "docs/manual/data/montana_extract"
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")
import warnings; warnings.simplefilter("ignore")
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, difflib, json, subprocess, tempfile
import rasterio
import yaml
from data_acquisition.catalog import build, describe, evidence
from data_acquisition.catalog import summary as summaries

OUT = ROOT / "output/manual/describe/training_pond"
REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())


def check(s):
    """Check a summary (a dict) and build it in memory; print what the pipeline says."""
    path = Path(tempfile.mkdtemp()) / "my_summary.yaml"
    path.write_text(yaml.safe_dump(copy.deepcopy(s), sort_keys=False))
    try:
        loaded = summaries.load(str(path))
    except ValueError as e:
        print("BLOCKS:\n", str(e).split("\n", 1)[1])
        return None
    catalog, report = build.build(SAMPLE, loaded)
    print("valid; build report:", {k: v for k, v in report.items() if v} or "empty")
    return catalog

**1. Evidence.** Gather the evidence and the prompt for the sample dataset into `output/manual/describe/training_pond`. For each file, what did code read? What did it read about the photos, and why so little?

In [ ]:
...  # your code here: describe prompt --root SAMPLE --folders "repo/*" --dataset training_pond

**2. The rules.** Read the rules in `prompt.md`. Which of them stop Claude from inventing values?

In [ ]:
...  # your code here (describe.RULES)

**3. By hand.** Write a summary for the sample from the evidence alone, starting from the skeleton; make it pass `check`; compare it with the reference summary.

In [ ]:
mine = {
    "schema": 1, "dataset": "training_pond", "title": "...",
    "authored": {"by": "A Student", "date": datetime.date.today()}, "reviewed": None,
    "source": {"kind": "dataone", "provider": "..."},
    "groups": [{"folder": "repo/doi_10.5555_training.pond", "record": {"path": "metadata.xml", "format": "eml"},
                "site": "training_pond", "files": [
                    # one entry per file: {"path": ..., "product": ..., "date": datetime.date(...)} or role + attach_to
                ]}],
}
check(mine)

**4. Review a draft.** `docs/manual/data/flawed_draft.yaml` passes the check but has four mistakes a reviewer should catch. Find them with the evidence.

In [ ]:
flawed = yaml.safe_load((ROOT / "docs/manual/data/flawed_draft.yaml").read_text())
check(flawed)
...  # your code here: compare its values with the evidence (headers, record, methods text)

**5. (Optional) Draft with Claude.** Run the prompt in a Claude Code session on the sample. The prompt writes into `data_acquisition/summaries/` and may add a site to `data_acquisition/sites.yaml`: delete the draft and undo any registry change afterwards (`git status`, `git restore`).

In [ ]:
# Done in a Claude Code session, not here. Afterwards:
print(subprocess.run(["git", "status", "--short"], cwd=ROOT, capture_output=True, text=True).stdout)

**6. Find it in the code.** Where are the drafting rules, and which function decides what evidence a raster gets?

In [ ]:
import inspect
...  # your code here